In [0]:
from pyspark.sql.functions import *  
from pyspark.sql.window import *
from pyspark.sql.types import *

In [0]:
df = spark.sql(
    """
    SELECT * FROM projetos_jr.camada_bronze.br_transacoes_bancarias
    """
) 

In [0]:
display(df)

# 1) Unicidade

- Verificar dados duplicados não esperados

## Resultado:
- A base de dados não possui valores duplicados em colunas que são PK/FK

In [0]:
def verificar_duplicatas(df, coluna):
    janela = Window.partitionBy(coluna)
    return(
        df.withColumn(
            "Qtd_duplicatas",
            count("*").over(janela)
        ).filter(col("Qtd_duplicatas") > 1)
    ) ''

In [0]:
display(verificar_duplicatas(df, "id_transacao"))

# 2) Completude
- Verificar a quantidade de dados nulos dentro do conjunto

## Resultado:
- A coluna cliente possui um total de 100 registros nulos, para não haver a perda desses dados, a estratégia utilizada foi aplicar "Não identificado"

In [0]:
df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns
]).display()

In [0]:
total_linhas = df.count()
df.select([
    round(
    sum(when(col("nome_cliente").isNull(), 1).otherwise(0)) / total_linhas * 100).alias("Percentual nulos")
]).display()

- Tratamento para dados que estão nulos

In [0]:
df = df.withColumn("nome_cliente", when(col("nome_cliente").isNull(), "Não identificado").otherwise(col("nome_cliente")))

display(df)


#3) Validade
- Verifica se os dados estão preenchidos conforme padrões esperados

In [0]:
df = df.withColumn("cpf", when(length(col("cpf")) != 11, "Fora do padrão").otherwise(col("cpf")))

display(df)

- Validade dos valores na coluna de idade

In [0]:
df.filter(
    col("idade") < 0
).display()

- Correção: Espaços / TAB no campo de Cidade

In [0]:
df = df.withColumn("cidade", trim("cidade"))

display(df)

- Correção: Campos de transações serem transformados para double

In [0]:
df = df.withColumn(
    "valor_transacao",
    regexp_replace(col("valor_transacao"), "R\\$\\s*", "").cast("float")
)

In [0]:
df.select(["valor_transacao"]).display()

In [0]:
df = df.withColumn("saldo_apos_transacao", col("saldo_apos_transacao").cast("float"))

In [0]:
df.select(["saldo_apos_transacao"]).display()

In [0]:
df = df.withColumn("data_transacao", col("data_transacao").cast("date"))

- Correção: Tipo do campo data

In [0]:
df.select(["data_transacao"]).display()

# Dados prontos para serem consumidos

- Salvos na camada ouro

In [0]:
catalog_name = 'projetos_jr'
database_name = 'camada_prta'
table_name = 'pr_transacoes_bancarias'

In [0]:
df.write.mode("overwrite").saveAsTable(f'{catalog_name}.{database_name}.{table_name}')

In [0]:
%sql 
SELECT *
FROM projetos_jr.camada_prata.pr_transacoes_bancarias